# Statistické testy na datové sadě Iris

Tento notebook obsahuje řešení následujících hypotéz:
1. Ověření hypotézy o průměrné délce kališního lístku (t-test).
2. Test mediánu délky kališního lístku (Wilcoxon).
3. Párové porovnání průměrů všech znaků (t-test).
4. Hotellingův T² test pro vektory průměrů.
5. Box M test pro kovarianční struktury.

Vizualizace: boxploty, scatter ploty, korelační heatmapy.


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import ttest_1samp, wilcoxon, ttest_ind
from sklearn.datasets import load_iris
from scipy.stats import f, chi2


In [14]:
iris = load_iris()
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df['species'] = pd.Categorical.from_codes(iris.target, iris.target_names)
# df.head()

global_mean  = df.iloc[:, :-1].mean()
print("Overall means:\n", global_mean)
global_median = df.iloc[:, :-1].median()
print("Overall medians:\n", global_median)
means = df.groupby('species').mean()
print("Means by species:\n", means)


Overall means:
 sepal length (cm)    5.843333
sepal width (cm)     3.057333
petal length (cm)    3.758000
petal width (cm)     1.199333
dtype: float64
Overall medians:
 sepal length (cm)    5.80
sepal width (cm)     3.00
petal length (cm)    4.35
petal width (cm)     1.30
dtype: float64
Means by species:
             sepal length (cm)  sepal width (cm)  petal length (cm)  \
species                                                              
setosa                  5.006             3.428              1.462   
versicolor              5.936             2.770              4.260   
virginica               6.588             2.974              5.552   

            petal width (cm)  
species                       
setosa                 0.246  
versicolor             1.326  
virginica              2.026  


C:\Users\sediva\AppData\Local\Temp\ipykernel_22152\3746157172.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  means = df.groupby('species').mean()


## 1. T-test pro průměrnou délku kališního lístku vs μ₀ = 5.0 cm

In [ ]:
mu0 = 5.0
for species in df['species'].unique():
    data = df[df['species'] == species]['sepal length (cm)']
    t_stat, p_val = ttest_1samp(data, mu0)
    print(f"{species}: t = {t_stat:.3f}, p-hodnota = {p_val:.4f}")


## 2. Wilcoxonův test pro medián délky kališního lístku vs 6.0 cm

In [ ]:
median_hyp = 6.0
for species in df['species'].unique():
    data = df[df['species'] == species]['sepal length (cm)']
    diff = data - median_hyp
    stat, p_val = wilcoxon(diff)
    print(f"{species}: W = {stat:.3f}, p-hodnota = {p_val:.4f}")


## 3. Párové t-testy pro všechny znaky mezi druhy

In [ ]:
species_list = df['species'].unique()
features = iris.feature_names
for feat in features:
    print(f"
Feature: {feat}")
    for i in range(len(species_list)):
        for j in range(i+1, len(species_list)):
            data1 = df[df['species'] == species_list[i]][feat]
            data2 = df[df['species'] == species_list[j]][feat]
            t_stat, p_val = ttest_ind(data1, data2)
            print(f"{species_list[i]} vs {species_list[j]}: t = {t_stat:.3f}, p = {p_val:.4f}")


In [ ]:
sns.boxplot(x='species', y='sepal length (cm)', data=df)
plt.title('Boxplot: Sepal Length podle druhu')
plt.show()


## 4. Hotellingův T² test pro vektory průměrů

In [ ]:
species_list = df['species'].unique()
p = len(features)
for i in range(len(species_list)):
    for j in range(i+1, len(species_list)):
        data1 = df[df['species'] == species_list[i]][features].values
        data2 = df[df['species'] == species_list[j]][features].values
        n1, n2 = data1.shape[0], data2.shape[0]
        mean_diff = data1.mean(axis=0) - data2.mean(axis=0)
        S1 = np.cov(data1, rowvar=False)
        S2 = np.cov(data2, rowvar=False)
        Sp = ((n1-1)*S1 + (n2-1)*S2)/(n1+n2-2)
        T2 = (n1*n2)/(n1+n2) * mean_diff.T @ np.linalg.inv(Sp) @ mean_diff
        F_stat = ((n1+n2-p-1)*T2)/((n1+n2-2)*p)
        df1, df2 = p, n1+n2-p-1
        p_val = 1 - f.cdf(F_stat, df1, df2)
        print(f"{species_list[i]} vs {species_list[j]}: T² = {T2:.2f}, F = {F_stat:.2f}, p = {p_val:.4f}")


## 5. Box M test pro kovarianční struktury

In [ ]:
for i in range(len(species_list)):
    for j in range(i+1, len(species_list)):
        data1 = df[df['species'] == species_list[i]][features].values
        data2 = df[df['species'] == species_list[j]][features].values
        n1, n2 = data1.shape[0], data2.shape[0]
        S1 = np.cov(data1, rowvar=False)
        S2 = np.cov(data2, rowvar=False)
        Sp = ((n1-1)*S1 + (n2-1)*S2)/(n1+n2-2)
        log_det_S1 = np.log(np.linalg.det(S1))
        log_det_S2 = np.log(np.linalg.det(S2))
        log_det_Sp = np.log(np.linalg.det(Sp))
        M = (n1-1)*log_det_S1 + (n2-1)*log_det_S2 - (n1+n2-2)*log_det_Sp
        p = len(features)
        g = 2
        c = (1 - ((2*p**2+3*p-1)/(6*(p+1)))*(1/(n1-1)+1/(n2-1)-1/(n1+n2-2)))
        Box_M = c*M
        df_chi = (p*(p+1)/2)*(g-1)
        p_val = chi2.sf(Box_M, df_chi)
        print(f"{species_list[i]} vs {species_list[j]}: Box M = {Box_M:.2f}, df = {df_chi}, p = {p_val:.4f}")
